# Summary

Load datasets to S3

In [1]:
import os, sys
import pandas as pd
import json

# AWS Python
import boto3

# Numantic utilities
utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

from utilities.text_cleaning import text_cleaning_tools as tct

api_configs = ApiAuthentication(client="Numantic")



## Read local data


In [2]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

# Read local data into Pandas dataframes
df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))

## Clean up docs

## Add some metadata for testing

In [3]:
source_type_map = {"https://enterthegungeon.fandom.com/wiki/Bullet_Kin": "gaming",
                   "https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1aksw5rm/c2%20-%20session%2050%20-%20underground.docx?rlkey=ioqwgkd14i5xk20i3fp38nzgs&e=1&dl=0": "gaming",
                   "https://bytes-and-nibbles.web.app/bytes/stici-note-part-1-planning-and-prototyping": "data_science",
                   "https://github.com/llmware-ai/llmware": "data_science",
                   "https://docs.marimo.io/recipes.html": "recipes",
                   "https://towardsdatascience.com/how-to-maximize-your-impact-as-a-data-scientist-3881995a9cb1": "data_science",
                   "https://ec.europa.eu/commission/presscorner/detail/en/QANDA_21_1683": "government",
                   "https://bg3.wiki/wiki/The_Emperor": "gaming",
                   "https://whattocook.substack.com/p/so-into-northern-spain": "recipes",
                   "https://dmtalkies.com/the-zone-of-interest-ending-explained-and-summary-2023-film/": "entertainment",
                   "https://www.loonyparty.com/about/policy-proposals/": "entertainment",
                   "https://timdettmers.com/2023/01/30/which-gpu-for-deep-learning/": "data_science",
                   "https://gleam.run/cheatsheets/gleam-for-python-users/": "data_science",
                   "https://towardsdatascience.com/gpt-from-scratch-with-mlx-acf2defda30e": "data_science",
                   "https://blog.reedsy.com/short-story/a3gstd/": "entertainment",
                   "http://www.chakoteya.net/DoctorWho/40-1.html": "entertainment",
                   "https://stardewvalleywiki.com/Version_History": "gaming",
                   "https://alanwake.fandom.com/wiki/Alan_Wake_2": "gaming",
                   "https://www.polygon.com/23691206/best-fantasy-books-sci-fi-2023": "entertainment",
                   "https://arxiv.org/pdf/2404.10981": "data_science"
                   }

df_docs["source_type"] = df_docs["source_url"].map(source_type_map)

In [4]:

tct_add = ("There are several giants. One is burly, grey-skinned and 20 feet tall. "
           "It wears heavy dark iron "
           "armour covered in metal thorns and is leaning against a chamber wall. "
           "It is carrying "
           "what looks to be two tower shields nearly as tall as itself, both bearing "
           "menacing spikes. The other is taller and in more mobile irons, clutching a "
           "dangerous-looking maul the size of Yasha. It too is leaning against a wall and looking disinterested.")


df_docs.loc[1, "text"] = "{}\n\n{}".format(df_docs.loc[1, "text"],
                                           tct_add)
# df_docs.loc[1, "text"]


In [5]:
print("Source documents")
display(df_docs.head())
display(pd.DataFrame(df_docs["source_type"].value_counts()))

print("Single-passage questions")
display(df_spqs.head())


Source documents


,index,source_url,text,source_type
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,gaming
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,gaming
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,data_science
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,data_science
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,recipes


,count
source_type,
data_science,7
gaming,5
entertainment,5
recipes,2
government,1


Single-passage questions


,document_index,question,answer
0,0,What do keybullet kin drop?,Keybullet kin drop a key upon death.
1,0,What kind of gun does the bandana bullet kin use?,The bandana bullet kin wields a machine pistol.
2,1,What do the giants look like?,"One giant is burly, grey-skinned, and 20 feet ..."
3,1,What happens on day 2?,"After a few miles of winding tunnel, you emerg..."
4,2,What were the requirements for the project?,The tool had the following requirements:\n- Ch...


## Load documents to S3

In [6]:

def process_and_upload(df, key_prefix='documents/'):
    # Prepare and load documents to S3 under a stable prefix.
    for idx in df.index:

        # Step 1: Get text and metadata fields
        raw_text = df.loc[idx, 'text']
        source_url = df.loc[idx, 'source_url']
        source_type = df.loc[idx, 'source_type']

        # Step 2: Annotate passage splits
        blocks = raw_text.split('\n\n')
        annotated_text = ""
        for i, block in enumerate(blocks, 1):
            annotated_text += f"[{i}]\n{block}\n\n"

        # Step 3: Create metadata
        metadata = {
            "metadataAttributes": {
                "source_url": source_url,
                "source_type": source_type,
                "document_index": int(idx),
            }
        }

        # Step 4: Create filenames/keys
        file_name = f"doc_{idx}.txt"
        metadata_name = f"{file_name}.metadata.json"
        text_key = f"{key_prefix}{file_name}"
        metadata_key = f"{key_prefix}{metadata_name}"

        # Step 5: Upload text and metadata files
        s3_client.put_object(
            Bucket=bucket,
            Key=text_key,
            Body=annotated_text.encode('utf-8')
        )

        s3_client.put_object(
            Bucket=bucket,
            Key=metadata_key,
            Body=json.dumps(metadata).encode('utf-8')
        )

    print(f"Successfully uploaded {len(df)} documents and metadata files to prefix '{key_prefix}'.")


In [7]:

### Step 1. Set up session
region_name = 'us-east-2'
session = boto3.Session(profile_name='ns-admin',
                        region_name=region_name)

### Step 2. Export documents to S3 in Bedrock-friendly format
s3_client = session.client('s3')
bucket = 'rag-search-tests'
s3_prefix = 'documents/'

### Step 3. Delete existing documents first
print(f"Deleting existing documents from {bucket}/{s3_prefix}...")
paginator = s3_client.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix=s3_prefix):
    if 'Contents' in page:
        for obj in page['Contents']:
            s3_client.delete_object(Bucket=bucket, Key=obj['Key'])
print("✓ Existing documents deleted")

### Step 4. Upload text and metadata files
process_and_upload(df=df_docs, key_prefix=s3_prefix)


Deleting existing documents from rag-search-tests/documents/...
✓ Existing documents deleted
Successfully uploaded 20 documents and metadata files to prefix 'documents/'.
